# F-09 Whisper 3-way 비교 (turbo 순정 vs final2 LoRA vs OpenAI whisper-1)

| 모델 | 방식 | 출처 |
|------|------|------|
| baseline | 로컬 추론 | `openai/whisper-large-v3-turbo` (HuggingFace) |
| final2 | 로컬 추론 (LoRA 어댑터) | Google Drive `checkpoints/whisper-senior/final2` |
| whisper-1 | OpenAI API (REST) | `https://api.openai.com/v1/audio/transcriptions` |

**목적**: Phase 1 시연 단계 STT(OpenAI `whisper-1` 클라우드) 채택이 자체 LoRA 파인튜닝 대비 시니어 발화에서 어느 정도 품질인지 확인.

**조정사항 (코랩 무료 T4 최적화)**:
- 평가 샘플 500 → 300
- batch size 8 → 4 (T4 16GB OOM 방지)
- 비교 모델 4개 → 3개 (final1/3 제외)
- OpenAI API 비용: 300 × 평균 5초 ≈ 25분 음성 × $0.006 ≈ **약 $0.15**

**실행 순서**: 셀 01 → 02 → 03 → 04

In [ ]:
# 셀 01 — 라이브러리 설치 + Drive 마운트 + OpenAI 키 + 경로 설정
!pip install -q transformers datasets peft accelerate evaluate jiwer librosa soundfile openai tqdm
!pip install -q "torchao>=0.16.0"

from google.colab import drive, userdata
from pathlib import Path
import os

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

DRIVE_ROOT     = Path('/content/drive/MyDrive/Dadam_dataSet')
DATASET_PATH   = DRIVE_ROOT / 'processed/senior_speech_v2'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints/whisper-senior'

# 로컬 추론 대상 (None = 순정, Path = LoRA 어댑터)
LOCAL_MODEL_PATHS = {
    'baseline': None,                          # 순정 turbo
    'final2'  : CHECKPOINT_DIR / 'final2',     # 2차 LoRA — 확정 모델
}

# OpenAI API 키 — Colab Secrets(좌측 자물쇠)에 OPENAI_API_KEY 등록 권장
# Secrets 미설정 시 input() 폴백 (한 번만 입력)
try:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print('OpenAI 키: Colab Secrets에서 로드 ✓')
except Exception:
    if not os.environ.get('OPENAI_API_KEY'):
        os.environ['OPENAI_API_KEY'] = input('OpenAI API 키 입력: ').strip()
    print('OpenAI 키: 환경변수에서 로드 ✓')

for name, path in LOCAL_MODEL_PATHS.items():
    if path is not None:
        print(f'{name}: {path}  존재={path.exists()}')
    else:
        print(f'{name}: 순정 모델')

print(f'데이터셋: {DATASET_PATH}  존재={DATASET_PATH.exists()}')

In [ ]:
# 셀 02 — 평가 함수 정의 (로컬 추론 + OpenAI API 두 경로)
import re
import io
import time
import torch
import evaluate
import soundfile as sf
from tqdm import tqdm
from openai import OpenAI
from torch.utils.data import DataLoader
from dataclasses import dataclass
from typing import Any
from datasets import load_from_disk
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel

# ── 공통 상수 ─────────────────────────────────────────────
MODEL_ID      = 'openai/whisper-large-v3-turbo'
EVAL_SAMPLES  = 300                # 500 → 300 (T4 무료 티어 시간·OpenAI 비용 절감)
BATCH_SIZE    = 4                  # 8 → 4 (T4 16GB OOM 방지)
NUM_BEAMS     = 1
SAMPLE_RATE   = 16_000
PUNCT_PATTERN = re.compile(r'[.?!,。、]')
cer_metric    = evaluate.load('cer')

def clean_text(text: str) -> str:
    """CER 계산 전 정규화: 구두점 제거 + 공백 제거"""
    return PUNCT_PATTERN.sub('', text).replace(' ', '').strip()

# ── 로컬 추론 (turbo·LoRA) ─────────────────────────────────
@dataclass
class EvalCollator:
    processor: Any
    def __call__(self, features):
        audio_arrays = [f['audio']['array'] for f in features]
        texts        = [f['text'] for f in features]
        inputs = self.processor(
            audio_arrays,
            sampling_rate=SAMPLE_RATE,
            return_tensors='pt',
            padding='max_length',
            max_length=480_000,
            truncation=True,
        )
        return {'input_features': inputs.input_features, 'texts': texts}

def evaluate_local(name, adapter_path, dataset):
    """로컬 GPU에서 turbo 또는 LoRA 어댑터 추론 + CER 계산"""
    print(f'\n[{name}] 로드 중...')
    processor  = WhisperProcessor.from_pretrained(MODEL_ID, language='Korean', task='transcribe')
    base_model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
    base_model.generation_config.language           = 'korean'
    base_model.generation_config.task               = 'transcribe'
    base_model.generation_config.forced_decoder_ids = None

    if adapter_path is not None:
        model = PeftModel.from_pretrained(base_model, str(adapter_path), is_trainable=False)
    else:
        model = base_model

    model = model.half().to('cuda').eval()

    eval_subset = dataset['validation'].select(range(EVAL_SAMPLES))
    loader      = DataLoader(eval_subset, batch_size=BATCH_SIZE, collate_fn=EvalCollator(processor))
    gen_kwargs  = dict(language='korean', task='transcribe', num_beams=NUM_BEAMS)

    all_preds, all_refs = [], []
    for batch in tqdm(loader, desc=f'{name} 추론', leave=False):
        input_feats = batch['input_features'].to('cuda', dtype=torch.float16)
        with torch.no_grad():
            pred_ids = model.generate(input_features=input_feats, **gen_kwargs)
        preds = processor.batch_decode(pred_ids, skip_special_tokens=True)
        all_preds.extend([clean_text(p) for p in preds])
        all_refs.extend( [clean_text(r) for r in batch['texts']])

    cer = cer_metric.compute(predictions=all_preds, references=all_refs)
    print(f'[{name}] CER: {cer:.4f}  ({cer * 100:.2f}%)')
    print('  샘플 3개:')
    for i in range(3):
        print(f'    정답: {all_refs[i]}')
        print(f'    예측: {all_preds[i]}')

    # VRAM 해제
    del model, base_model
    torch.cuda.empty_cache()
    return cer, all_preds, all_refs

# ── OpenAI API 추론 (whisper-1) ──────────────────────────
def evaluate_openai_api(name, model_id, dataset, max_retries=3):
    """OpenAI Audio Transcriptions API로 평가 (REST batch)"""
    print(f'\n[{name}] OpenAI API 호출 시작...')
    client = OpenAI()  # OPENAI_API_KEY 환경변수 자동 사용
    eval_subset = dataset['validation'].select(range(EVAL_SAMPLES))

    all_preds, all_refs = [], []
    failed_count = 0
    for item in tqdm(eval_subset, desc=f'{name} API', leave=False):
        # numpy array → WAV bytes (API는 파일 객체 요구)
        buf = io.BytesIO()
        sf.write(buf, item['audio']['array'], SAMPLE_RATE, format='WAV')
        buf.seek(0)
        buf.name = 'audio.wav'

        # rate limit 대비 재시도 (지수 백오프)
        pred_text = ''
        for attempt in range(max_retries):
            try:
                resp = client.audio.transcriptions.create(
                    model=model_id,
                    file=buf,
                    language='ko',
                )
                pred_text = resp.text
                break
            except Exception as e:
                if attempt < max_retries - 1:
                    wait = 2 ** attempt
                    print(f'  재시도 {attempt+1}/{max_retries} ({wait}s 대기): {e}')
                    time.sleep(wait)
                    buf.seek(0)  # 파일 포인터 리셋
                else:
                    failed_count += 1
        all_preds.append(clean_text(pred_text))
        all_refs.append(clean_text(item['text']))

    if failed_count > 0:
        print(f'  ⚠️ API 실패 {failed_count}건 (빈 문자열로 처리됨 → CER 보수적 추정)')

    cer = cer_metric.compute(predictions=all_preds, references=all_refs)
    print(f'[{name}] CER: {cer:.4f}  ({cer * 100:.2f}%)')
    print('  샘플 3개:')
    for i in range(3):
        print(f'    정답: {all_refs[i]}')
        print(f'    예측: {all_preds[i]}')
    return cer, all_preds, all_refs

dataset = load_from_disk(str(DATASET_PATH))
print('데이터셋 로드 완료')
print(dataset)

In [ ]:
# 셀 03 — 3개 모델 순차 평가 (로컬 2개 → API 1개)
# 로컬 모델 먼저 평가하여 VRAM 사용 → 해제. 그 후 API 호출은 GPU 무관.
results = {}

# 로컬 추론 (baseline, final2)
for name, path in LOCAL_MODEL_PATHS.items():
    cer, _, _ = evaluate_local(name, path, dataset)
    results[name] = round(cer * 100, 2)

# OpenAI API 추론 (whisper-1)
cer, _, _ = evaluate_openai_api('whisper-1', 'whisper-1', dataset)
results['whisper-1'] = round(cer * 100, 2)

print('\n' + '=' * 45)
print('3-way 비교 결과 요약')
print('=' * 45)
for name, cer in results.items():
    print(f'{name:<12} {cer:>7.2f}%')
print('=' * 45)

In [ ]:
# 셀 04 — 결과 요약 표 출력 (노션 복사용)
notes = {
    'baseline' : '순정 turbo (파인튜닝 없음)',
    'final2'   : '2차 LoRA — 시니어 음성 확정 튜닝',
    'whisper-1': 'OpenAI 호스팅 API ($0.006/분)',
}

baseline_cer = results.get('baseline', None)
col1, col2, col3, col4 = 12, 10, 16, 36
header = f"{'모델':<{col1}} {'CER (%)':>{col2}}  {'baseline 대비':>{col3}}  {'비고'}"
sep    = '-' * (col1 + col2 + col3 + col4 + 6)

print(sep)
print(header)
print(sep)

for name in ['baseline', 'final2', 'whisper-1']:
    if name not in results:
        print(f'{name:<{col1}} {"측정 안됨":>{col2}}')
        continue
    cer = results[name]
    diff_str = '-' if (baseline_cer is None or name == 'baseline') else f'{cer - baseline_cer:+.2f}%p'
    print(f'{name:<{col1}} {cer:>{col2}.2f}%  {diff_str:>{col3}}  {notes[name]}')

print(sep)
best = min(results, key=results.get)
print(f'  최저 CER: {best}  ({results[best]:.2f}%)')
print(sep)